Part A: Churn model

In [1]:
#Preparing data
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

df = pd.read_csv("../data/clean/gaming_clean.csv")
num = ["age","daily_hours","spend_inr","signup_year"]
cat = ["gender","state","platform","genre"]

X = df[num + cat + ["esports_participant"]].copy()
X["esports_participant"] = X.esports_participant.astype(int)
y = df.churn_risk.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [3]:
#Training and evaluating
pre = ColumnTransformer([
    ("num", StandardScaler(), num + ["esports_participant"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
model = Pipeline([("prep", pre), ("clf", LogisticRegression(max_iter=500))])
model.fit(X_train, y_train)

print("Baseline accuracy (always 'no churn'):", round(1 - y_test.mean(), 4))
print("Model accuracy:", round(accuracy_score(y_test, model.predict(X_test)), 4))
print("AUC:", round(roc_auc_score(y_test, model.predict_proba(X_test)[:,1]), 4))


Baseline accuracy (always 'no churn'): 0.7803
Model accuracy: 0.7803
AUC: 0.5006


CONCLUSION: The model achieves an AUC of 0.50, no better than random guessing, and its accuracy equals the always-"no churn" baseline. The available variables carry no predictive signal for churn risk. Real churn data would need behavioural features such as session frequency, days since last login or purchase history.

Part B: Player segments (K-means)

In [4]:
#Choosing K
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sample = df.sample(100000, random_state=42)[["age","daily_hours","spend_inr"]]
Z = StandardScaler().fit_transform(sample)

for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=5, random_state=42).fit(Z)
    print(k, round(km.inertia_), round(silhouette_score(Z[:10000], km.labels_[:10000]), 3))

2 224885 0.266
3 179369 0.269
4 130356 0.294
5 103953 0.305
6 94803 0.301


In [5]:
#Profile the segmet
km = KMeans(n_clusters=4, n_init=5, random_state=42).fit(Z)
sample["segment"] = km.labels_
sample.groupby("segment").agg(players=("age","size"), avg_age=("age","mean"),
                              avg_hours=("daily_hours","mean"),
                              avg_spend=("spend_inr","mean")).round(1)

,players,avg_age,avg_hours,avg_spend
segment,,,,
0,33052,40.5,1.6,3811.6
1,11634,30.1,6.9,3741.8
2,27963,23.0,1.6,1678.7
3,27351,22.4,1.6,5749.9
